- references
    - https://verl.readthedocs.io/en/latest/algo/rollout_corr.html
        - https://verl.readthedocs.io/en/latest/algo/rollout_corr_math.html
    - https://yingru.notion.site/When-Speed-Kills-Stability-271211a558b7808d8b12d403fd15edda
- $\pi_\theta$, $\pi_{old}$, $\pi_{\text{rollout}}$, $\pi_{ref}$
    - openrlhf: vllm as the rollout/generation engine
- 数据流
    - 生成本 step batch -> 立刻算 old logprob/value/advantage -> update actor -> sync weights to rollout。
- RL 训练里的 on-policy 缓解，但这里有两层问题需要分开：
    - PPO 本身缓解的是：同一批 rollout 数据被多轮 mini-batch update 反复使用后，当前 policy 和采样 policy 之间产生 drift。
        - PPO ratio 解决 pi_theta 和 pi_old 的 near-on-policy update 问题。
        - pi_theta vs pi_old: PPO 正常要处理的 update drift。
    - verl 的 decoupled / rollout correction 还额外缓解：rollout engine 和 trainer actor 不是同一个物理计算路径，导致 rollout policy 和 training-side old policy 不一致。
        - verl 的 decoupled / rollout correction 还处理 pi_old 和 pi_rollout 的 rollout-training mismatch。
        - pi_old vs pi_rollout: rollout/training decoupling 带来的 behavior mismatch。
- verl bypass & decoupled mode
    - bypass：$ \frac{\pi_\theta}{\pi_{\text{rollout}}} $
    - decoupled mode：$\frac{\pi_\theta}{\pi_{\text{rollout}}}=\frac{\pi_\theta}{\pi_{\text{old}}}\cdot\frac{\pi_{\text{old}}}{\pi_{\text{rollout}}}$
- 为什么 bypass 不足
    - PPO update 开始时，当前 actor 应该等于 update 前的 old actor：$\pi_\theta=\pi_{\text{old}}$
    - 所以标准 PPO ratio 在 update 刚开始时应该满足：$\frac{\pi_\theta(a_t \mid s_t)}{\pi_{\text{old}}(a_t \mid s_t)}=1$,这保证 PPO update 是从 ratio 等于 1 的位置开始，然后随着 optimizer step 推动 $\pi_\theta$ 偏离 $\pi_{\text{old}}$。
    - 但 bypass 下 ratio 是：$\frac{\pi_\theta(a_t \mid s_t)}{\pi_{\text{rollout}}(a_t \mid s_t)}$
        - 如果 rollout stale 或计算路径不一致，那么即使 update 尚未开始，也可能有：$\frac{\pi_{\text{old}}(a_t \mid s_t)}{\pi_{\text{rollout}}(a_t \mid s_t)}\neq1$
    - 这意味着 PPO 在第一步就已经把 rollout-training mismatch 混进了 PPO ratio。
- 一个具体数值例子
    - 假设某个 token 在 rollout engine 下的概率是：$\pi_{\text{rollout}}(a_t \mid s_t)=0.40$
    - 但 trainer actor 在 update 前重新 forward 得到：$\pi_{\text{old}}(a_t \mid s_t)=0.60$
    - update 刚开始时：$\pi_\theta=\pi_{\text{old}}$
    - 如果使用 decoupled PPO ratio：$r_t(\theta)=\frac{\pi_\theta(a_t \mid s_t)}{\pi_{\text{old}}(a_t \mid s_t)}=\frac{0.60}{0.60}=1.0$
        - 这符合 PPO 的预期。
    - 但如果使用 bypass：$r_t^{\text{bypass}}(\theta)=\frac{\pi_\theta(a_t \mid s_t)}{\pi_{\text{rollout}}(a_t \mid s_t)}=\frac{0.60}{0.40}=1.5$
        - 如果 PPO clip range 是：$\epsilon=0.2$，那么 clip 区间是：$\left[1-\epsilon,1+\epsilon\right]=\left[0.8,1.2\right]$
        - 此时：$1.5>1.2$
    - 也就是说，update 刚开始，这个 token 就已经因为 rollout mismatch 被推到 clip 区间外。
    - 这不是标准 PPO 希望处理的 policy update drift，而是 rollout engine 与 trainer actor 的 mismatch 被错误塞进了 PPO ratio。

### bypass 和 decoupled 在代数上等价吗

- 从纯代数看，对同一个 token、同一个 state/history、同一个 action，只要分母非零，就有恒等式：

$$
\frac{\pi_\theta(a_t \mid s_t)}{\pi_{\text{rollout}}(a_t \mid s_t)}=
\frac{\pi_\theta(a_t \mid s_t)}{\pi_{\text{old}}(a_t \mid s_t)}\cdot
\frac{\pi_{\text{old}}(a_t \mid s_t)
}{\pi_{\text{rollout}}(a_t \mid s_t)
}$$

用简写就是：

$$
\frac{\pi_\theta}{\pi_{\text{rollout}}}=
\frac{\pi_\theta}{\pi_{\text{old}}}\cdot\frac{\pi_{\text{old}}}{\pi_{\text{rollout}}
}
$$

所以在数学恒等式层面，二者当然等价。

但是在 PPO 算法层面，bypass 和 decoupled 不等价。原因是 PPO 不是只计算一个裸 ratio，而是对 ratio 做 clipping、masking、normalization、rejection 或 truncated importance sampling 等非线性处理。

bypass 把整个 ratio 放进 PPO clipping：

$$
r_t^{\text{bypass}}(\theta)=\frac{\pi_\theta}{\pi_{\text{rollout}}}=r_t^{\text{PPO}}(\theta)\cdot
\rho_t^{\text{rollout}}
$$

于是 clip 看到的是完整乘积：

$$
\operatorname{clip}
\left(
r_t^{\text{PPO}}(\theta)
\cdot
\rho_t^{\text{rollout}},
1-\epsilon,
1+\epsilon
\right)
$$

decoupled 则把 PPO trust-region ratio 和 rollout correction ratio 分开：

$$
r_t^{\text{PPO}}(\theta)=
\frac{
\pi_\theta
}{
\pi_{\text{old}}
}
$$

$$
\rho_t^{\text{rollout}}=
\frac{
\pi_{\text{old}}
}{
\pi_{\text{rollout}}
}
$$

PPO clipping 主要作用在：

$$
r_t^{\text{PPO}}(\theta)
$$

而 rollout correction 作用在另一个位置，例如作为 rollout importance weight、rejection mask、truncated IS weight 或 debug metric。

因此一般有：

$$
\operatorname{clip}
\left(
r_t^{\text{PPO}}(\theta)
\cdot
\rho_t^{\text{rollout}},
1-\epsilon,
1+\epsilon
\right)
\neq
\rho_t^{\text{rollout}}
\cdot
\operatorname{clip}
\left(
r_t^{\text{PPO}}(\theta),
1-\epsilon,
1+\epsilon
\right)
$$

这就是“代数上可分解，但算法上不等价”的核心原因。

假设：

$$
r_t^{\text{PPO}}(\theta)
=
\frac{
\pi_\theta
}{
\pi_{\text{old}}
}
=
1.0
$$

并且：

$$
\rho_t^{\text{rollout}}
=
\frac{
\pi_{\text{old}}
}{
\pi_{\text{rollout}}
}
=
1.5
$$

那么 bypass ratio 是：

$$
r_t^{\text{bypass}}(\theta)
=
r_t^{\text{PPO}}(\theta)
\cdot
\rho_t^{\text{rollout}}
=
1.0 \cdot 1.5
=
1.5
$$

如果 $\epsilon = 0.2$，bypass 的 PPO clip 会看到：

$$
\operatorname{clip}
\left(
1.5,
0.8,
1.2
\right)
=
1.2
$$

但 decoupled 的 PPO clip 看到的是：

$$
\operatorname{clip}
\left(
1.0,
0.8,
1.2
\right)
=
1.0
$$

rollout mismatch 的 $1.5$ 会在另一个机制中处理，而不是直接让 PPO trust-region ratio 从 1.5 开始。

所以：

```text
bypass:
  把 rollout mismatch 和 PPO update drift 合成一个 ratio，再一起 clip。

decoupled:
  先让 PPO ratio 从 pi_theta / pi_old 开始，保证 update 起点是 1；
  再单独处理 pi_old / pi_rollout 的 rollout mismatch。
```

### $\pi_{\text{rollout}}$

> verl 对 $\pi_{old}$ 有进一步的区分：$\pi_\text{rollout}$, $\pi_\text{old}$

- 在经典 PPO 理论里，$\pi_{\text{old}} = \pi_{\text{behavior}}$，也就是采样 rollout 数据的那个模型（$\pi_{\text{behavior}}$ 确实就是刚刚与环境完成交互、负责收集当前批次数据的 $\pi_{\text{old}}$；而与之相对的 Target Policy，则是当前正处于内部优化循环中、其神经网络参数正在被梯度反向传播不断更新的最新策略 $\pi_{\theta}$）。因此如果是一个严格同步、同一套模型实现、同一份权重、同一精度、采完马上训的 PPO，确实不需要额外引入 $\pi_{\text{rollout}}$。这时：
$$
\pi_{\text{rollout}} = \pi_{\text{old}}
$$
- verl 讨论的训练--推理不一致，关键点就在于：工程系统里的 rollout policy 不一定等于 training 侧重新计算出来的 old policy。
- 在 verl 的语境里可以这样理解：$\pi_{\text{rollout}}$ 是真正生成 token 的 policy。它可能来自 vLLM/SGLang rollout engine，用推理优化 kernel、低精度、不同 batch 组织、不同 attention backend，甚至异步场景下可能是稍早版本的权重。
- $\pi_{\text{old}}$ 是训练侧 PPO update 开始前，用 actor 重新对这批 rollout token 计算 logprob 得到的 policy（用于计算）。它通常来自训练 actor，例如 FSDP/TP actor，使用训练图、训练精度和训练侧 forward。
- $\pi_\theta$是正在被梯度更新的当前 policy。
- 所以在 verl 的 decoupled mode 里，实际拆成两个 drift：$\pi_{\text{rollout}} \rightarrow \pi_{\text{old}} \rightarrow \pi_\theta$
    - 第一段是rollout engine 和训练 actor 的不一致，第二段是PPO 多步更新里 old policy 到 current policy 的变化。

> 为什么会有不同：权重文件“看起来是同一个 checkpoint”
- 推理引擎和训练引擎不同：rollout 可能用 vLLM/SGLang，training 用 PyTorch/FSDP/Megatron。kernel、attention 实现、position handling、padding、mask、logits processor 都可能导致 logprob 细微差异。
- 数值精度不同：rollout 可能是 BF16、FP8、量化 KV cache 或其他推理优化；training 侧可能是 BF16/FP32 mixed precision。token 概率对 logits 很敏感，小差异会累积成 sequence-level 大差异。
- 异步或 stale rollout (experimental fully async 路径明确体现 stale)：分布式 RL 里 rollout worker 采样时的权重可能落后于 trainer 当前准备更新的 actor 权重。
    - 于是 rollout 数据真实来自旧版本：$\pi_{\text{rollout}} = \pi_{\theta_{t-k}}$
    - 但训练侧 recompute 的 old logprob 可能对应：$\pi_{\text{old}} = \pi_{\theta_t}$
- tokenization / chat template / stop condition / logprob 计算路径差异
    - 如果 rollout 侧和 training 侧对 prompt、response mask、special token、EOS、padding 的处理不同，表面上是“同一模型”，实际计算的条件概率已经不是同一个对象。
- MoE router / parallelism / non-determinism
    - MoE 模型里 router replay、expert dispatch、并行切分等如果不能完全复现 rollout 时的路径，training 侧重新算出的 token logprob 也可能偏离 rollout 真实概率。
    - MoE 模型里，一个 token 经过 MoE layer 时不是经过所有 FFN expert，而是 router 先算 expert logits，再选 top-k experts。简化为：$h' = \sum_{e \in TopK(router(h))} w_e\cdot E_e(h)$, 如果 top-k expert 变了，哪怕 token id、模型权重“看起来一样”，实际执行的 FFN 权重集合也变了。于是 hidden state 变，后续 logits 变，最终 token logprob 也变。
    - router top-k 可能有近 tie，比如 expert 7 分数 1.0000，expert 13 分数 0.9998。BF16/FP8、不同 kernel、不同 reduce 顺序就可能交换 top-k。
  - rollout 用 SGLang/vLLM，training 用 Megatron，router 实现、fused MoE kernel、token dispatcher、expert parallel 切分可能不同。
  - expert dispatch 会把 token 按 expert 分组、permute、all-to-all 到 EP ranks，再 gather 回来。不同 packing/sequence parallel/dynamic batch 顺序下，必须保证 (batch, seq, layer, topk) 的路由记录和 token 对齐。
  - 一旦早期 token hidden state 变，后续 autoregressive KV/cache 和 teacher-forced logprob 也会继续偏移。

> sequence level
- 看一个极简二分类 softmax。rollout 侧 logits：$z^{r} = [0, 0]$
    - 采到 token 0，则：$p_r(token_0)=0.5$
    - training 侧因为精度、kernel、router、并行顺序等原因，logits 只发生很小变化：$z^{old} = [-0.02, 0.02]$ 这时：$p_{old}(token_0)=\frac{e^{-0.02}}{e^{-0.02}+e^{0.02}}\approx 0.4900$
    - 单 token ratio 是：$\rho_t=\frac{p_{old}}{p_r}\approx \frac{0.49}{0.50}=0.98$
    - 单步看只差 2%。但 sequence ratio 是乘积：$\rho_{\text{seq}}=\prod_{t=1}^{T}\rho_t$（verl 的代码大量在 log-space 里处理 ratio $\exp(\sum_t \log \rho_t)$）
-   如果每个 token 都是 0.98：

| 长度 | sequence ratio |
|---:|---:|
| 128 | $0.98^{128}\approx 0.075$ |
| 512 | $0.98^{512}\approx 3.2\times 10^{-5}$ |
| 2048 | $0.98^{2048}\approx 1.1\times 10^{-18}$ |

- 反过来如果每步是 1.02：$1.02^{512}\approx 2.5\times 10^4$
- LLM 不是“一个动作”，而是几百到几千个 token 的动作序列。每步 logprob 偏差只有 0.02，512 token 后就是：$\exp(512\times -0.02)\approx 3.6\times 10^{-5}$，这足以让 sequence-level IS 权重崩掉。

对自回归语言模型，给定 prompt \(x\)，response 为：

$$
y = (y_1, y_2, \ldots, y_T)
$$

语言模型不是直接给整个 \(y\) 一个黑盒概率，而是按 token 条件概率链式分解：

$$
\pi(y \mid x)
=
\prod_{t=1}^{T}
\pi(y_t \mid x, y_{<t})
$$

现在有两个 policy：训练侧旧 policy 和 rollout policy。

训练侧旧 policy 对整段 response 的概率是：

$$
\pi_{\text{old}}(y \mid x)
=
\prod_{t=1}^{T}
\pi_{\text{old}}(y_t \mid x, y_{<t})
$$

rollout policy 对同一段 response 的概率是：

$$
\pi_{\text{rollout}}(y \mid x)
=
\prod_{t=1}^{T}
\pi_{\text{rollout}}(y_t \mid x, y_{<t})
$$

sequence-level importance ratio 定义为：

$$
\rho_{\text{seq}}
=
\frac{
\pi_{\text{old}}(y \mid x)
}{
\pi_{\text{rollout}}(y \mid x)
}
$$

把两边的自回归分解代进去：

$$
\rho_{\text{seq}}
=
\frac{
\prod_{t=1}^{T}
\pi_{\text{old}}(y_t \mid x, y_{<t})
}{
\prod_{t=1}^{T}
\pi_{\text{rollout}}(y_t \mid x, y_{<t})
}
$$

因为分子和分母都是同一组 token 位置上的连乘，所以可以逐 token 相除：

$$
\rho_{\text{seq}}
=
\prod_{t=1}^{T}
\frac{
\pi_{\text{old}}(y_t \mid x, y_{<t})
}{
\pi_{\text{rollout}}(y_t \mid x, y_{<t})
}
$$

定义单 token ratio：

$$
\rho_t
=
\frac{
\pi_{\text{old}}(y_t \mid x, y_{<t})
}{
\pi_{\text{rollout}}(y_t \mid x, y_{<t})
}
$$

于是：

$$
\rho_{\text{seq}}
=
\prod_{t=1}^{T}
\rho_t
$$

这就是 sequence ratio 为什么是连乘。

> logprob 形式

实际代码里不会直接乘概率，而是用 logprob。因为连乘容易 underflow/overflow。

单条序列的 logprob 是：

$$
\log \pi(y \mid x)
=
\sum_{t=1}^{T}
\log \pi(y_t \mid x, y_{<t})
$$

因此 sequence log-ratio 是：

$$
\log \rho_{\text{seq}}
=
\log
\frac{
\pi_{\text{old}}(y \mid x)
}{
\pi_{\text{rollout}}(y \mid x)
}
$$

也就是：

$$
\log \rho_{\text{seq}}
=
\log \pi_{\text{old}}(y \mid x)
-
\log \pi_{\text{rollout}}(y \mid x)
$$

代入自回归 logprob：

$$
\log \rho_{\text{seq}}
=
\sum_{t=1}^{T}
\log \pi_{\text{old}}(y_t \mid x, y_{<t})
-
\sum_{t=1}^{T}
\log \pi_{\text{rollout}}(y_t \mid x, y_{<t})
$$

合并为逐 token 差：

$$
\log \rho_{\text{seq}}
=
\sum_{t=1}^{T}
\left(
\log \pi_{\text{old}}(y_t \mid x, y_{<t})
-
\log \pi_{\text{rollout}}(y_t \mid x, y_{<t})
\right)
$$

而单 token log-ratio 是：

$$
\log \rho_t
=
\log \pi_{\text{old}}(y_t \mid x, y_{<t})
-
\log \pi_{\text{rollout}}(y_t \mid x, y_{<t})
$$

所以：

$$
\log \rho_{\text{seq}}
=
\sum_{t=1}^{T}
\log \rho_t
$$

两边取指数：

$$
\rho_{\text{seq}}
=
\exp
\left(
\sum_{t=1}^{T}
\log \rho_t
\right)
$$

又因为：

$$
\exp
\left(
\sum_{t=1}^{T}
\log \rho_t
\right)
=
\prod_{t=1}^{T}
\rho_t
$$

所以最终得到：

$$
\rho_{\text{seq}}
=
\prod_{t=1}^{T}
\rho_t
$$

```python
log_ratio_sum = masked_sum(log_ratio, response_mask, axis=-1).unsqueeze(-1)
raw_rollout_is_weights = exp(clamp(log_ratio_sum, -20, 20)).expand_as(log_ratio)
```

$$
\log \rho_t
=
\log \pi_{\text{old}}(y_t \mid x, y_{<t})
-
\log \pi_{\text{rollout}}(y_t \mid x, y_{<t})
$$

$$
\log \rho_{\text{seq}}
=
\sum_{t=1}^{T}
\log \rho_t
$$

$$
\rho_{\text{seq}}
=
\exp
\left(
\log \rho_{\text{seq}}
\right)
$$

即：

$$
\rho_{\text{seq}}
=
\exp
\left(
\sum_{t=1}^{T}
\left[
\log \pi_{\text{old}}(y_t \mid x, y_{<t})
-
\log \pi_{\text{rollout}}(y_t \mid x, y_{<t})
\right]
\right)
$$

#### why ppo/grpo $\pi_{old}$

PPO clipping 控制的是当前更新不要离 old policy 太远：

$$
r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)}
$$

这个 $\pi_{\text{old}}$ 是 PPO 的 proximal anchor。但如果数据其实来自 $\pi_{\text{rollout}}$，那么严格来说，期望分布不对。你正在用：

$\tau \sim \pi_{\text{rollout}}$ 的数据，去估计本来希望在 $\pi_{\text{old}}$ 下的 PPO objective。所以需要一个 correction：

$$
w_t = \frac{\pi_{\text{old}}(a_t|s_t)}{\pi_{\text{rollout}}(a_t|s_t)}
$$

于是可以把问题拆开：

$$
\frac{\pi_\theta}{\pi_{\text{rollout}}}=\frac{\pi_\theta}{\pi_{\text{old}}}
\cdot
\frac{\pi_{\text{old}}}{\pi_{\text{rollout}}}
$$

- $\frac{\pi_\theta}{\pi_{\text{old}}}$：PPO 原本的 clipping ratio
- $\frac{\pi_{\text{old}}}{\pi_{\text{rollout}}}$：rollout correction / IS weight

什么时候它们可以视为一致: $\pi_{\text{rollout}} \approx \pi_{\text{old}}$

- rollout 和 training 用同一套 forward 实现
- 同一份权重
- 同一精度或数值差异极小
- 同步 rollout，采完立刻训练
- logprob 计算路径完全一致
- tokenizer、mask、EOS、padding 完全一致

### $\pi_\theta$, $\pi_{old}$, $\pi_{ref}$

> 生成本 step batch -> 立刻算 old logprob/value/advantage -> update actor -> sync weights to rollout。

- `ActorRolloutRefWorker`
    - This worker can be instantiated as a standalone actor or a standalone rollout or a standalone reference policy or a hybrid engine based on the config.rollout
    - 逻辑上
        - Policy 是 Role.ActorRollout 同一个 worker 里既负责 actor update，也负责 rollout
        - Ref 是 Role.RefPolicy （仅在开 KL reward / KL loss 时才创建）
    - 物理部署上
        - ActorRollout 和 RefPolicy 都映射到 global_pool （Colocation (共生/同驻)）
            - `self.mapping[Role.ActorRollout] = global_pool_id`
            - `self.mapping[Role.Critic] = global_pool_id`
            - `self.mapping[Role.RefPolicy] = "global_pool"`
    - 计算
        - Actor 与 Rollout 的切换 (Hybrid Engine)：`fsdp_workers.py`
            - rollout_mode: 切换到生成模式
            - trainer_mode(): 切换到训练模式
        - RefPolicy 通常是一个冻结的模型，不需要反向传播。为了节省显存给 Actor 训练，RefPolicy 采用了 FSDP CPU Offload 策略。
            - 当需要计算 Ref Log Prob 时，FSDP 机制会自动将需要的参数分片（Shard）流式传输（Stream）到 GPU 进行前向计算，计算完后立即释放 GPU 显存。
- 计算时机
    - 同一 batch 多轮 PPO 更新导致“批内陈旧”：π_old 在 batch 开头算一次，但 π_θ 会随 ppo_epochs/mini-batch 迭代偏移，后续更新天然更 off-policy。

- 在普通 decoupled PPO 路径里，物理上通常不会常驻一个单独的 pi_old 模型副本；pi_old 主要被物化成一张 old_log_probs tensor，作为后续 PPO mini update 的固定 anchor。($\pi_{old}$跟$\pi_\theta$ 物理上不是同时存在的两种模型)
    - rollout engine 生成数据，可能同时返回 rollout_log_probs，这是 pi_rollout 对生成 token 的 logprob。
    - trainer 在 actor 更新前调用 actor 重新 forward 一次，算出 old_log_probs。
        - computed once per data batch, serves as stable reference during mini-batch updates
    - 把 old_log_probs 存进 batch。
    -  开始 PPO mini-batch / multi-epoch 更新，此时 actor 权重不断变化，当前 forward 得到的是 pi_theta。
    -  loss 里用当前 log_prob - old_log_probs 得到 PPO ratio。
        -  exp(log_prob - old_log_prob)
- $\pi_{\text{old}}$：不是一个常驻冻结模型，而是“更新开始前 actor 权重对这批 token 的 logprob 快照”。 

### mismatch

> 在重要性采样（Importance Sampling, IS）的理论框架下，分母 $\pi_{\text{old}}$ 必须严格等于产生数据的那个分布（Behavior Policy），否则 IS 估计本身就是错的。

$$
\frac{\pi_\theta}{\pi_{\text{old}}}
$$
- 虽然 Actor 和 Rollout 模型的权重（Weights）是同步的，但由于底层实现的差异，它们对同一个输入的输出概率分布并不完全一致：
    - 计算精度与算子差异：vLLM/SGLang 为了极致的推理速度，使用了 PagedAttention、FlashInfer 等特定 CUDA 核心，且可能在 KV Cache 上使用了不同的精度（如 FP8/Int8）或累加方式，这与 PyTorch FSDP 前向传播时的计算逻辑存在数值误差。
        - 真正的 Behavior Policy ($\mu$)：是 vLLM。数据 $x$ 是从 $\pi_{\text{vLLM}}(x)$ 采样出来的。
        - 理想的 Behavior Policy ($\pi_{\text{old}}$)：我们原本以为 $\mu$ 就是上一轮的 $\pi_{\text{actor\old}}$。
        - mismatch 的后果$ \mu(x) = \pi_{\text{vLLM}}(x) \neq \pi_{\text{actor\old}}(x) $
    - Tokenization/Chat Template：有时训练和推理的 Tokenizer 处理方式（如 padding、special tokens）也可能有细微差别。
    - RolloutCorrectionConfig
        - 1. Policy mismatch: Rollout policy (e.g., vLLM BF16) vs Training policy (e.g., FSDP FP32)
        - 2. Model update staleness: Rollout data collected from older policy checkpoints
        - 3. General off-policy scenarios: Any distribution shift between data collection and training
- 在标准的 On-Policy 算法（如 PPO/GRPO）中，我们假设数据是由当前的“老策略” $\pi_{\text{old}}$ 产生的。
    - 理想情况：数据由 $\pi_{\text{actor}}$ 产生，更新时计算 $\frac{\pi_{\text{new}}}{\pi_{\text{actor}}}$。
    - 实际情况（Mismatch）：数据实际上是由 $\pi_{\text{vllm}}$ 产生的。
    - 通常的做法（Hybrid Engine 模式）是让 Actor 在这些数据上重算（Recompute）一遍 **old_log_probs**，即我们假装数据是由 $\pi_{\text{actor}}$ 产生的。但这忽略了一个事实：数据的分布（采样概率）是 $\pi_{\text{vllm}}$ 决定的。
        - 既然文本是 vLLM 生成的，那么这个样本之所以存在于你的 Batch 里，是因为 vLLM 觉得它概率高，而不是因为 Actor 觉得它概率高。
    - 当 $\pi_{\text{vllm}}$ 和 $\pi_{\text{actor}}$ 差异较大时（KL 散度变大），你的采样分布与你的优化目标分布不一致，这就引入了隐式的 Off-Policy 问题。这会导致重要性采样权重（Importance Sampling Ratios）估计不准，进而造成训练不稳定甚至崩塌。
- `verl/trainer/ppo/rollout_corr_helper.py` & `verl/trainer/config/algorithm.py: RolloutCorrectionConfig`

### cases & examples

> vLLM 和 Actor 发生了 Mismatch（比如因为量化精度导致 logits 略有不同）。“数据是由 vLLM（推理引擎） 决定的”。

- vLLM (实际生成者)：
    - 认为 "好" 的概率是 90% ($0.9$)。
    - 因为它概率高，vLLM 很容易就采样到了 "好" 这个字。
    - 结果：数据集中多了一条样本 "今天天气很好"。
- Actor (PyTorch FSDP, 训练者)：
    - 由于计算精度差异，Actor 重新计算 "好" 字的概率时，算出只有 10% ($0.1$)。
- 在 PPO 算法中，我们需要计算重要性采样比率（Importance Sampling Ratio）：
    $$ \text{Ratio} = \frac{\pi_{\text{new}}(\text{"好"})}{\pi_{\text{old}}(\text{"好"})} $$
    - 错误的假设（忽略事实）：你直接用 Actor 重算的值作为 $\pi_{\text{old}}$。
    $$ \text{Ratio} = \frac{\pi_{\text{new}}(0.1 \approx)}{\mathbf{0.1}} \approx 1.0 $$
    - 正确的做法（承认事实）：你应该用 vLLM 当时采样时的真实概率作为 $\pi_{\text{old}}$。
    $$ \text{Ratio} = \frac{\pi_{\text{new}}(0.1 \approx)}{\mathbf{0.9}} \approx 0.11 $$

### VeRL

- Bypass Mode： 用于效率，因为它跳过 old logprob recompute；
    - 代码会直接把 old_log_probs 设为 rollout_log_probs。这意味着在后续所有的 PPO 计算中，$\pi_{\text{old}}$ 就是 $\pi_{\text{vllm}}$。
        - $ \text{Ratio} = \frac{\pi_\theta}{\pi_{\text{rollout}}} $
        - 然后进行 `from verl.trainer.ppo.rollout_corr_helper import apply_rollout_correction`
    - 否则 `Recompute old_log_probs`

符号定义
- $\pi_{\theta}$: 当前正在优化的策略网络（Current Policy），参数为 $\theta$。
- $\pi_{\text{old}}$: 在本次 PPO 更新迭代（Iteration）开始时，Actor 网络的快照（Snapshot/Anchor Policy）。它是本次 Trust Region 的基准。
- $\pi_{\text{rollout}}$: 实际生成数据的策略（Behavior Policy），即推理引擎（vLLM/SGLang）。由于 Mismatch，$\pi_{\text{rollout}} \neq \pi_{\text{old}}$。
- $A(s, a)$: 优势函数（Advantage），通常基于 $\pi_{\text{rollout}}$ 采样的数据估算。

标准的 PPO 目标函数假设数据是由 $\pi_{\text{old}}$ 产生的：
$$
L^{\text{PPO}}(\theta) = \mathbb{E}_{(s,a) \sim \pi_{\text{old}}} \left[ \min \left( r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t \right) \right]
$$
其中 $r_t(\theta) = \frac{\pi_{\theta}(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)}$。
Mismatch 的核心矛盾：
实际上数据 $(s,a)$ 是从 $\pi_{\text{rollout}}$ 采样的，而非 $\pi_{\text{old}}$。如果我们忽略这一点，直接用 Actor 重算的 $\pi_{\text{old}}(a|s)$ 作为分母，会导致 $r_t(\theta)$ 的估计严重偏离真实的重要性采样比率，因为 $\pi_{\text{old}}(a|s)$ 可能远小于 $\pi_{\text{rollout}}(a|s)$。

Bypass Mode (绕过模式)
核心思想：承认 $\pi_{\text{rollout}}$ 才是真正的“旧策略”（Old Policy）。直接将 $\pi_{\text{old}}$ 替换为 $\pi_{\text{rollout}}$。
Objective:
$$
L^{\text{Bypass}}(\theta) = \mathbb{E}_{(s,a) \sim \pi_{\text{rollout}}} \left[ \min \left( r^{\text{bypass}}_t(\theta) A_t, \text{clip}(r^{\text{bypass}}_t(\theta), 1-\epsilon, 1+\epsilon) A_t \right) \right]
$$
其中 Ratio 定义为：
$$
r^{\text{bypass}}_t(\theta) = \frac{\pi_{\theta}(a_t|s_t)}{\pi_{\text{rollout}}(a_t|s_t)}
$$
优点：简单直接，理论上无偏（Unbiased），因为分母正是数据生成分布。
缺点：PPO 的 Trust Region 约束变成了针对 $\pi_{\text{rollout}}$ 的约束。如果 $\pi_{\text{rollout}}$ 和当前的 $\pi_{\theta}$ 差异过大（例如 inference engine 做了严重的量化或近似），可能导致约束过紧或优化目标偏离原始意图。

Decoupled Mode (解耦模式 / Correction Mode)
核心思想：保留 $\pi_{\text{old}}$ 作为 Trust Region 的锚点（Anchor），但引入重要性采样权重（Importance Sampling Weight）$w_t$ 来修正数据分布的偏移。
我们将目标函数重写为关于 $\pi_{\text{rollout}}$ 的期望：
$$
\mathbb{E}_{x \sim \pi_{\text{old}}} [f(x)] = \mathbb{E}_{x \sim \pi_{\text{rollout}}} \left[ \frac{\pi_{\text{old}}(x)}{\pi_{\text{rollout}}(x)} f(x) \right]
$$
在 verl 的实现中，这个修正系数 $w_t$ 被定义为：
$$
w_t = \frac{\pi_{\text{old}}(a_t|s_t)}{\pi_{\text{rollout}}(a_t|s_t)}
$$
注意：这里的 $\pi_{\text{old}}$ 是 Actor 在训练开始前的快照，$\pi_{\text{rollout}}$ 是 vLLM 的输出。
Objective (带 IS 修正的 PPO):
$$
L^{\text{Decoupled}}(\theta) = \mathbb{E}_{(s,a) \sim \pi_{\text{rollout}}} \left[ w_t \cdot \min \left( r^{\text{std}}_t(\theta) A_t, \text{clip}(r^{\text{std}}_t(\theta), 1-\epsilon, 1+\epsilon) A_t \right) \right]
$$
其中 Ratio 依然是标准的 PPO Ratio：
$$
r^{\text{std}}_t(\theta) = \frac{\pi_{\theta}(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)}
$$
展开来看 (不考虑 Clip)：
$$
L \approx \mathbb{E}_{\pi_{\text{rollout}}} \left[ \frac{\pi_{\text{old}}}{\pi_{\text{rollout}}} \cdot \frac{\pi_{\theta}}{\pi_{\text{old}}} \cdot A \right] = \mathbb{E}_{\pi_{\text{rollout}}} \left[ \frac{\pi_{\theta}}{\pi_{\text{rollout}}} \cdot A \right]
$$

$$
\nabla L \propto - \mathbb{E} [ \underbrace{w_t}_{\text{rollout\_is\_weights}} \cdot r_t(\theta) \cdot A_t ]
$$
- rollout_is_weights 对应的正是 $w_t$。
- 通过 $w_t$ 可以检测异常值（Rejection Sampling）。

#### $\pi_{ref}$

- $\pi_{ref}$ 的 logprob 也是 先算好，再存在 batch 里。这一点和 $\pi_{old}$ 类似。
    - 之后 PPO update 用的是这个 ref_log_prob tensor，不需要在每个 mini-batch 里重新跑 reference model。
- offload / CPU offload (把暂时不用、或者不需要一直放在 GPU 上的数据，从 GPU 显存搬到别的地方，通常是 CPU 内存，以降低显存占用。)
    - For models larger than 7B, it’s recommended to turn on offload for ref by default

```
fsdp_config:
    forward_only: True
```
在 FSDP 实现里，如果是 forward_only ref，verl 会对 ref 使用 CPU offload：
```
if self.engine_config.forward_only:
  cpu_offload = CPUOffload(offload_params=True)
```

或 FSDP2 下使用： `CPUOffloadPolicy(pin_memory=True)`

| 层面 | 含义 | 常见场景 |
|---|---|---|
| model 级 | 整个模型的参数整体不常驻 GPU，或者整个 engine 进入 offload 状态 | 手动 model.to("cpu")、rollout sleep、简单模型卸载 |
| parameter / shard 级 | FSDP 把参数切成 shard，每个 rank 只持有一部分，forward 前 all-gather，需要后释放 | FSDP / ZeRO |
| layer / module 级 | 当前 layer 计算前把需要的参数搬到 GPU，算完再释放或 offload | FSDP CPUOffload、ZeRO-Infinity、activation offload |
| optimizer-state 级 | Adam 的 m/v 等状态放 CPU，需要 optimizer step 时搬运或在 CPU 更新 | optimizer offload |
| activation 级 | forward 的中间激活不全留 GPU，需要 backward 时重算或从 CPU 拿回 | activation offload |

verl 的 ref model 如果走 FSDP，并且是 forward_only=True，代码里会用：`CPUOffload(offload_params=True)`或者 FSDP2 的：`CPUOffloadPolicy(pin_memory=True)`，这不是简单地“把整个 ref model 一次性搬上 GPU，算完整个 batch，再整个搬回 CPU”。

```
更准确地说，FSDP 会管理被 shard / wrap 的 module 参数：

进入某个 FSDP-wrapped module forward
      ↓
把该 module 当前需要的参数 shard / full param 准备到 GPU
      ↓
执行 forward
      ↓
forward 后 reshard / release / offload
```

| 符号 | 物理上是什么 | 什么时候加载 / 初始化 | 什么时候计算 | 产物 | 是否更新 |
|---|---|---|---|---|---|
| $\pi_\theta$ | trainable actor 模型，带 optimizer | actor_rollout_wg.init_model() 初始化 actor；resume 时从 checkpoint 恢复 | PPO update 的每个 mini-batch / micro-batch forward | 当前 actor 的 log_probs，用于 PPO ratio | 会被 optimizer step 更新 |
| $\pi_{old}$ | 通常不是单独模型，而是 rollout batch 上预先算好的 old_log_probs | 不单独加载 | rollout 之后、PPO update 之前，用当前 actor forward 一次算出 | 固定的 old_log_probs tensor | batch 内固定不变 |
| $\pi_{ref}$ | frozen reference policy，可是独立 ref worker，也可能复用 actor 并关闭 LoRA adapter | 需要 reference policy 时初始化；ref_in_actor=False 时单独 ref_policy_wg.init_model() | 每个 batch 上计算 ref_log_prob | KL reference 的 ref_log_prob | 不训练 |
| $\pi_{rollout}$ | rollout / inference engine 里的模型副本，例如 vLLM、SGLang、HF rollout | rollout worker / replica 初始化；随后从 actor 同步权重 | generate 阶段采样 token；可选返回 rollout logprob | response tokens，可选 rollout_log_probs | 不直接训练，只从 actor 同步 |

 情况 | 数据来自 | update ratio | 性质 |
|---|---|---|---|
| 严格 on-policy | $\pi_\theta$ | 不需要 IS，或每步重新采样 | 最干净，但成本最高 |
| 经典 PPO | $\pi_{\text{old}}$ | $\pi_\theta / \pi_{\text{old}}$ | near-on-policy |
| verl decoupled PPO | $\pi_{\text{rollout}}$ | PPO 用 $\pi_\theta / \pi_{\text{old}}$，rollout correction 用 $\pi_{\text{old}} / \pi_{\text{rollout}}$ | near-on-policy + rollout mismatch correction |
| bypass | $\pi_{\text{rollout}}$ | $\pi_\theta / \pi_{\text{rollout}}$ | 当 $\pi_{\text{rollout}} \approx \pi_{\text{old}}$ 时合理，否则会污染 PPO anchor |

bypass 不是永远错误。它在以下条件近似成立时是合理的：

$$
\pi_{\text{rollout}}
\approx
\pi_{\text{old}}
$$

也就是：

- rollout engine 权重新鲜。
- rollout logprob 精确表示 behavior policy。
- rollout engine 和 trainer actor 的 logprob 语义一致。
- 采样处理、mask、temperature、stop token、MoE routing 等路径一致。
- 序列不太长，ratio 方差可控。

如果这些条件不成立，bypass 就不足。